# Tech Challenge — Fase 2
## Notebook 01 — Ingestão Batch: camadas Bronze e Silver

O Compromisso Nacional Criança Alfabetizada estabelece que toda criança deve estar alfabetizada até
o fim do 2º ano do ensino fundamental. A Pesquisa Alfabetiza Brasil (Inep, 2023) definiu o corte de
743 pontos na escala do Saeb como o limiar da alfabetização, e a meta nacional é 80% até 2030.

Para responder "quais municípios estão fora da rota da meta", é preciso integrar fontes que hoje
estão separadas: o indicador por município e por UF, as metas nacionais, estaduais e municipais, os
dados territoriais e os microdados de alunos. É o que este notebook faz, em cinco etapas: setup,
Bronze (ingestão sem transformar), qualidade, Silver (limpeza e integração) e monitoramento.

```
  CSVs (Base dos Dados) ─┐
  API IBGE Localidades  ─┼─ batch ──►  BRONZE  ──►  SILVER  ──►  GOLD
  Microdados de alunos  ─┘             (bruto)     (integrado)   (analítico)
  Eventos de avaliação  ── streaming ──┘                             │
        (Notebook 02)                                        Athena / BI / ML
```

Ordem de execução: 01 (batch) → 02 (streaming) → 03 (Gold + cloud) → 04 (análises e IA).

---
## Etapa 1 — Setup

O notebook detecta se está no Colab ou local e resolve os caminhos sozinho. O data lake local usa a
mesma estrutura de pastas do S3 (`bronze/`, `silver/`, `gold/`), então portar para a nuvem é só
trocar o prefixo do caminho. O log segue o mesmo formato dos jobs Glue da aula.

In [ ]:
# ============================================================
# ETAPA 1.1 — DEPENDÊNCIAS
# ============================================================
# pyarrow para ler e gravar parquet, tabulate para as tabelas de qualidade e
# basedosdados para consultar o BigQuery (opcional)

!pip install pyarrow pandas requests tabulate colorama --quiet
!pip install basedosdados --quiet 2>/dev/null || echo "basedosdados indisponivel - o notebook usa o fallback documentado na Etapa 2.4"

print("Dependências prontas.")

In [ ]:
# ============================================================
# ETAPA 1.2 — IMPORTS, DETECÇÃO DE AMBIENTE E RAIZ DO PROJETO
# ============================================================
import os
import sys
import json
import time
import shutil
import logging
import subprocess
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

# ─── Identificação do ambiente ───────────────────────────────
IN_COLAB  = "google.colab" in sys.modules
REPO_NAME = "tech-challenge-fase2"
# ⚠️ Troque pela URL do repositório do grupo antes de rodar no Colab:
REPO_URL  = "https://github.com/SEU-USUARIO/tech-challenge-fase2.git"


def localizar_raiz_projeto():
    """Descobre a raiz do repositório.

    Regra: a raiz é o primeiro diretório (subindo a partir do cwd) que contém
    'data/landing'. No Colab, se o repositório ainda não estiver presente,
    ele é clonado — assim o notebook é reprodutível a partir do zero.
    """
    for candidato in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidato / "data" / "landing").is_dir():
            return candidato

    if IN_COLAB:
        destino = Path("/content") / REPO_NAME
        if not destino.exists():
            print(f"Clonando o repositório em {destino} ...")
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(destino)], check=True)
        return destino

    raise RuntimeError(
        "Raiz do projeto não encontrada. Abra este notebook de dentro do repositório "
        "(pasta que contém 'data/landing') ou ajuste REPO_URL para clonar no Colab."
    )


PROJECT_ROOT = localizar_raiz_projeto()
os.chdir(PROJECT_ROOT)          # todos os caminhos daqui pra frente são relativos à raiz
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Ambiente     : {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Raiz projeto : {PROJECT_ROOT}")

In [ ]:
# ============================================================
# ETAPA 1.3 — DATA LAKE EM CAMADAS (espelho local do S3)
# ============================================================
# Na nuvem cada camada fica num bucket separado (SOR, SOT, SPEC), com regra de
# ciclo de vida diferente. Aqui a estrutura de pastas é a mesma, só que local,
# então para portar basta trocar o prefixo do caminho.

LANDING  = Path("data/landing")                                  # arquivos originais baixados
LAKE     = Path(os.environ.get("TC2_LAKE", "data/lake"))         # raiz do data lake local
BRONZE   = LAKE / "bronze"
SILVER   = LAKE / "silver"
GOLD     = LAKE / "gold"
QUALITY  = LAKE / "quality"
MONITOR  = LAKE / "monitoring"

for caminho in (BRONZE, SILVER, GOLD, QUALITY, MONITOR):
    caminho.mkdir(parents=True, exist_ok=True)

# ─── Espelho da nomenclatura em nuvem (usado nos logs e nos jobs Glue) ───
AWS_ACCOUNT_ID = os.environ.get("AWS_ACCOUNT_ID", "000000000000")
AWS_REGION     = "us-east-1"
S3_PREFIX = {
    "bronze": f"s3://{AWS_ACCOUNT_ID}-tc2-data-sor/bronze",
    "silver": f"s3://{AWS_ACCOUNT_ID}-tc2-data-sot/silver",
    "gold":   f"s3://{AWS_ACCOUNT_ID}-tc2-data-spec/gold",
}

# ─── Identificação da execução (rastreabilidade fim a fim) ───
RUN_ID         = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")
ANO_ING, MES_ING, DIA_ING = INGESTION_DATE.split("-")

# ─── Log estruturado no mesmo formato dos jobs AWS Glue ───
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%SZ",
    force=True,
)
log = logging.getLogger("pipeline.batch")

log.info("=" * 70)
log.info(f"PIPELINE  : tc2-alfabetizacao-batch")
log.info(f"RUN_ID    : {RUN_ID}")
log.info(f"DATA LAKE : {LAKE.resolve()}")
log.info(f"BRONZE    : {S3_PREFIX['bronze']}")
log.info(f"SILVER    : {S3_PREFIX['silver']}")
log.info("=" * 70)

---
### 1.4 — Catálogo de fontes e dicionário

O desafio pede a integração de seis entidades. Antes de escrever transformação, registramos o grão e
a chave de cada uma — é o que evita join errado depois.

| Entidade | Origem | Grão | Chave |
|---|---|---|---|
| UF | API IBGE + Base dos Dados | 1 linha por UF | `id_uf` / `sigla_uf` |
| Município | API IBGE + Base dos Dados | 1 linha por município | `id_municipio` |
| Meta Brasil | CSV | ano × rede | `ano` + `rede` |
| Meta UF | CSV | ano × UF × rede | `ano` + `sigla_uf` + `rede` |
| Meta Município | CSV | ano × município × rede | `ano` + `id_municipio` + `rede` |
| Alunos | BigQuery ou simulado | 1 linha por aluno | `id_aluno` |

Somam-se a elas as duas tabelas de resultado do indicador (município e UF), que são o fato central.

**Dicionário de `rede`.** As tabelas de indicador trazem `rede` como código numérico e as de meta
como texto (`Municipal`, `Pública`). Sem harmonizar isso a integração não fecha. Usamos o dicionário
do Inep (`0` Total, `2` Estadual, `3` Municipal, `5` Pública) e validamos contra os próprios dados na
Etapa 4: a taxa da rede Municipal bate 1:1 com a tabela de metas municipais.

In [ ]:
# ============================================================
# ETAPA 1.4 — CATÁLOGO DE FONTES E REGRAS DE NEGÓCIO
# ============================================================

# ─── Parâmetros oficiais da política pública ───
PONTO_CORTE_SAEB   = 743    # Pesquisa Alfabetiza Brasil (Inep, 2023)
META_NACIONAL_2030 = 80.0   # % de crianças alfabetizadas ao fim do 2º ano
SERIE_AVALIADA     = 2      # 2º ano do ensino fundamental

# ─── Dicionário da coluna 'rede' ───
# Os CSVs do indicador trazem rede como número e os de meta como texto, então
# é preciso um de-para. Se o dicionário oficial já foi baixado do BigQuery, ele é
# usado; caso contrário vale o mapeamento abaixo, conferido na Etapa 4.
DIM_REDE_PADRAO = {0: "Total", 1: "Federal", 2: "Estadual",
                   3: "Municipal", 4: "Privada", 5: "Pública"}

caminho_dicionario = LANDING / "dicionario.csv"
if caminho_dicionario.exists():
    dicionario = pd.read_csv(caminho_dicionario)
    oficial = {int(linha.chave): str(linha.valor)
               for linha in dicionario[dicionario["nome_coluna"] == "rede"].itertuples()
               if str(linha.chave).strip().isdigit()}
    DIM_REDE = {**DIM_REDE_PADRAO, **oficial}
    ORIGEM_DIM_REDE = f"dicionario oficial do dataset ({len(oficial)} códigos)"
else:
    DIM_REDE = DIM_REDE_PADRAO
    ORIGEM_DIM_REDE = "mapeamento do Inep, validado contra os dados na Etapa 4"

# Nas tabelas de meta a rede vem como texto — normalizamos para o mesmo código.
REDE_TEXTO_PARA_CODIGO = {v.lower(): k for k, v in DIM_REDE.items()}
REDE_TEXTO_PARA_CODIGO["publica"] = REDE_TEXTO_PARA_CODIGO.get("pública", 5)

print(f"Dicionário de rede: {ORIGEM_DIM_REDE}")
print(f"  {DIM_REDE}\n")

# ─── Catálogo de fontes (contrato de ingestão) ───
CATALOGO_FONTES = [
    {"entidade": "indicador_municipio", "origem": "basedosdados/csv", "modo": "batch",
     "arquivo": "br_inep_avaliacao_alfabetizacao_municipio.csv",
     "chave": ["ano", "id_municipio", "serie", "rede"]},
    {"entidade": "indicador_uf", "origem": "basedosdados/csv", "modo": "batch",
     "arquivo": "br_inep_avaliacao_alfabetizacao_uf.csv",
     "chave": ["ano", "sigla_uf", "serie", "rede"]},
    {"entidade": "meta_brasil", "origem": "basedosdados/csv", "modo": "batch",
     "arquivo": "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv",
     "chave": ["ano", "rede"]},
    {"entidade": "meta_uf", "origem": "basedosdados/csv", "modo": "batch",
     "arquivo": "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv",
     "chave": ["ano", "sigla_uf", "rede"]},
    {"entidade": "meta_municipio", "origem": "basedosdados/csv", "modo": "batch",
     "arquivo": "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv",
     "chave": ["ano", "id_municipio", "rede"]},
    {"entidade": "dim_uf_ibge", "origem": "api_ibge_localidades", "modo": "batch",
     "arquivo": None, "chave": ["id"]},
    {"entidade": "dim_municipio_ibge", "origem": "api_ibge_localidades", "modo": "batch",
     "arquivo": None, "chave": ["id"]},
    {"entidade": "aluno", "origem": "bigquery|simulado", "modo": "batch+streaming",
     "arquivo": None, "chave": ["id_aluno"]},
]

df_catalogo = pd.DataFrame(CATALOGO_FONTES)
print(f"Fontes catalogadas: {len(df_catalogo)}\n")
display(df_catalogo)

---
### 1.5 — Monitoramento

Criamos um `PipelineMonitor` que registra, por etapa: duração, linhas de entrada/saída, linhas
rejeitadas e status. No fim gera um JSON com o resumo da execução — o mesmo insumo que, na AWS,
alimentaria o CloudWatch. Gravamos a classe em `src/utils/monitoring.py` com `%%writefile`, para que
os jobs Glue do Notebook 03 possam reaproveitá-la.

In [ ]:
%%writefile src/utils/monitoring.py
"""Guarda as metricas de cada etapa da pipeline: tempo, linhas e erros."""
import json
import logging
import time
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

log = logging.getLogger("pipeline.monitor")


class PipelineMonitor:

    def __init__(self, pipeline, run_id, output_dir="data/lake/monitoring",
                 sla_etapa_segundos=120.0, limite_rejeicao_pct=5.0):
        self.pipeline = pipeline
        self.run_id = run_id
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.sla_etapa_segundos = sla_etapa_segundos
        self.limite_rejeicao_pct = limite_rejeicao_pct
        self.inicio = time.perf_counter()
        self.etapas = []
        self.alertas = []

    @contextmanager
    def etapa(self, nome, camada="-", sla_segundos=None):
        # usar com "with": dentro do bloco a gente preenche linhas_entrada/linhas_saida
        contexto = {"etapa": nome, "camada": camada, "linhas_entrada": 0,
                    "linhas_saida": 0, "linhas_rejeitadas": 0, "bytes_gerados": 0,
                    "detalhe": ""}
        t0 = time.perf_counter()
        status, erro = "OK", None
        try:
            yield contexto
        except Exception as exc:
            status = "FALHA"
            erro = f"{type(exc).__name__}: {exc}"
            self.registrar_alerta("CRITICAL", f"Falha na etapa '{nome}': {erro}")
            raise
        finally:
            duracao = round(time.perf_counter() - t0, 3)
            registro = dict(contexto)
            registro["status"] = status
            registro["erro"] = erro
            registro["duracao_segundos"] = duracao
            registro["timestamp"] = datetime.now(timezone.utc).isoformat(timespec="seconds")

            entrada = registro["linhas_entrada"] or registro["linhas_saida"]
            registro["throughput_linhas_s"] = round(entrada / duracao, 1) if duracao else 0.0
            registro["taxa_rejeicao_pct"] = (
                round(registro["linhas_rejeitadas"] / entrada * 100, 2) if entrada else 0.0)
            self.etapas.append(registro)

            if status == "OK":
                log.info("[MONITOR] %-38s | %-6s | %7.3fs | in=%s out=%s rej=%s",
                         nome, camada, duracao, registro["linhas_entrada"],
                         registro["linhas_saida"], registro["linhas_rejeitadas"])

            limite = sla_segundos or self.sla_etapa_segundos
            if duracao > limite:
                self.registrar_alerta("WARNING", f"Etapa '{nome}' levou {duracao}s (SLA {limite}s)")
            if registro["taxa_rejeicao_pct"] > self.limite_rejeicao_pct:
                self.registrar_alerta(
                    "WARNING",
                    f"Etapa '{nome}' rejeitou {registro['taxa_rejeicao_pct']}% das linhas")

    def registrar_alerta(self, severidade, mensagem):
        self.alertas.append({"severidade": severidade, "mensagem": mensagem,
                             "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds")})
        if severidade == "CRITICAL":
            log.error("[ALERTA:%s] %s", severidade, mensagem)
        elif severidade == "WARNING":
            log.warning("[ALERTA:%s] %s", severidade, mensagem)
        else:
            log.info("[ALERTA:%s] %s", severidade, mensagem)

    def resumo(self):
        colunas = ["etapa", "camada", "status", "duracao_segundos", "linhas_entrada",
                   "linhas_saida", "linhas_rejeitadas", "taxa_rejeicao_pct",
                   "throughput_linhas_s"]
        if not self.etapas:
            return pd.DataFrame(columns=colunas)
        return pd.DataFrame(self.etapas)[colunas]

    def metricas_gerais(self):
        df = self.resumo()
        return {
            "pipeline": self.pipeline,
            "run_id": self.run_id,
            "duracao_total_segundos": round(time.perf_counter() - self.inicio, 3),
            "etapas_executadas": len(df),
            "etapas_com_falha": int((df["status"] == "FALHA").sum()) if len(df) else 0,
            "linhas_processadas": int(df["linhas_saida"].sum()) if len(df) else 0,
            "linhas_rejeitadas": int(df["linhas_rejeitadas"].sum()) if len(df) else 0,
            "alertas": len(self.alertas),
            "alertas_criticos": sum(1 for a in self.alertas if a["severidade"] == "CRITICAL"),
        }

    def salvar(self):
        dados = self.metricas_gerais()
        dados["gerado_em"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
        dados["etapas"] = self.etapas
        dados["alertas"] = self.alertas

        destino = self.output_dir / f"run_{self.pipeline}_{self.run_id}.json"
        destino.write_text(json.dumps(dados, indent=2, ensure_ascii=False), encoding="utf-8")
        log.info("[MONITOR] Relatorio salvo em %s", destino)
        return destino

    def imprimir_painel(self):
        m = self.metricas_gerais()
        print("\n" + "=" * 78)
        print(f"  PAINEL DE EXECUCAO - {m['pipeline']} | run {m['run_id']}")
        print("=" * 78)
        print(self.resumo().to_string(index=False))
        print("-" * 78)
        print(f"  Duracao total ....... {m['duracao_total_segundos']}s")
        print(f"  Etapas .............. {m['etapas_executadas']} ({m['etapas_com_falha']} com falha)")
        print(f"  Linhas processadas .. {m['linhas_processadas']}")
        print(f"  Linhas rejeitadas ... {m['linhas_rejeitadas']}")
        print(f"  Alertas ............. {m['alertas']} ({m['alertas_criticos']} criticos)")
        if self.alertas:
            print("-" * 78)
            for a in self.alertas:
                print(f"  [{a['severidade']}] {a['mensagem']}")
        print("=" * 78)

In [ ]:
# ============================================================
# ETAPA 1.5 — INICIANDO O MONITOR DA EXECUÇÃO
# ============================================================
from src.utils.monitoring import PipelineMonitor

monitor = PipelineMonitor(
    pipeline="tc2-alfabetizacao-batch",
    run_id=RUN_ID,
    output_dir=MONITOR,
    sla_etapa_segundos=120,      # SLA por etapa; se estourar, gera alerta
    limite_rejeicao_pct=5.0,     # acima disso, a fonte passa a ser suspeita
)
print("Monitor ativo — todas as etapas a seguir serão medidas.")

---
## Etapa 2 — Camada BRONZE

A Bronze guarda os dados como vieram da fonte, sem `dropna` nem `rename`. Só acrescentamos colunas
de linhagem (prefixo `_`):

| Coluna | Serve para |
|---|---|
| `_ingestion_timestamp` / `_ingestion_date` | quando o dado entrou |
| `_source` / `_source_format` | de onde veio e em que formato |
| `_entity` | qual entidade do catálogo originou a linha |
| `_pipeline_run_id` | qual execução gerou a linha |
| `_record_hash` | impressão digital do registro, para detectar mudanças |

Decisões desta camada: **Parquet + Snappy** (colunar, comprimido, com schema), **particionamento por
data de ingestão** (permite reprocessar uma janela) e **escrita idempotente** (rodar de novo no mesmo
dia sobrescreve a partição, não duplica).

In [ ]:
%%writefile src/utils/lakeio.py
"""Funcoes para ler e gravar os parquets do data lake.

A coluna de particao fica no nome da pasta, e na leitura o pandas devolve ela
como Categorical, o que quebra operacao simples como .max(). Os tipos nullable
(Int16, Int32) tambem nao sobrevivem a ida e volta.
"""
from pathlib import Path

import pandas as pd


def preparar_particoes(df, particoes):
    """Converte as colunas de particao para tipos que aguentam a ida e volta."""
    dados = df.copy()
    for coluna in particoes or []:
        if pd.api.types.is_integer_dtype(dados[coluna]):
            dados[coluna] = dados[coluna].astype("int64")
        else:
            dados[coluna] = dados[coluna].astype(str)
    return dados


def ler_parquet(caminho):
    """Le um parquet (arquivo ou pasta particionada) e desfaz o Categorical."""
    df = pd.read_parquet(Path(caminho), engine="pyarrow")
    for coluna in df.columns:
        if isinstance(df[coluna].dtype, pd.CategoricalDtype):
            df[coluna] = df[coluna].astype(df[coluna].cat.categories.dtype)
    return df

In [ ]:
# ============================================================
# ETAPA 2.1 — MOTOR DE GRAVAÇÃO DA BRONZE
# ============================================================
# Uma função concentra a gravação na bronze: metadados, hash do registro,
# particionamento e registro no manifesto.

from src.utils.lakeio import ler_parquet, preparar_particoes

MANIFESTO_BRONZE = []


def tamanho_diretorio(caminho):
    """Soma o tamanho (bytes) de todos os arquivos de um diretório."""
    caminho = Path(caminho)
    if caminho.is_file():
        return caminho.stat().st_size
    return sum(f.stat().st_size for f in caminho.rglob("*") if f.is_file())


def formatar_bytes(n):
    for unidade in ("B", "KB", "MB", "GB"):
        if n < 1024 or unidade == "GB":
            return f"{n:.1f} {unidade}"
        n /= 1024


def hash_registros(df, colunas=None):
    """Hash de cada linha, para depois identificar o que mudou.

    hash_pandas_object é vetorizado. Com apply + hashlib levaria minutos nos
    3,8 milhões de registros de aluno.
    """
    base = df[colunas] if colunas else df
    return pd.util.hash_pandas_object(base, index=False).astype("uint64").astype(str)


def gravar_bronze(df, entidade, origem, formato_origem):
    """Grava na bronze sem mexer no conteúdo."""
    dados = df.copy()
    colunas_negocio = [c for c in dados.columns if not c.startswith("_")]

    # ── Linhagem ──
    dados["_ingestion_timestamp"] = RUN_ID
    dados["_ingestion_date"]      = INGESTION_DATE
    dados["_source"]              = origem
    dados["_source_format"]       = formato_origem
    dados["_entity"]              = entidade
    dados["_pipeline_run_id"]     = RUN_ID
    dados["_record_hash"]         = hash_registros(dados, colunas_negocio)

    # ── Partições físicas (data de ingestão) ──
    dados["ano_ingestao"] = ANO_ING
    dados["mes_ingestao"] = MES_ING
    dados["dia_ingestao"] = DIA_ING

    destino = BRONZE / entidade
    if destino.exists():
        shutil.rmtree(destino)          # idempotência: reprocessar não duplica
    dados = preparar_particoes(dados, ["ano_ingestao", "mes_ingestao", "dia_ingestao"])
    dados.to_parquet(
        destino,
        engine="pyarrow",
        compression="snappy",
        partition_cols=["ano_ingestao", "mes_ingestao", "dia_ingestao"],
        index=False,
    )

    bytes_gravados = tamanho_diretorio(destino)
    MANIFESTO_BRONZE.append({
        "entidade": entidade,
        "origem": origem,
        "formato_origem": formato_origem,
        "linhas": len(dados),
        "colunas": len(dados.columns),
        "bytes": bytes_gravados,
        "caminho_local": str(destino),
        "caminho_s3": f"{S3_PREFIX['bronze']}/{entidade}/ano_ingestao={ANO_ING}/mes_ingestao={MES_ING}/dia_ingestao={DIA_ING}/",
        "run_id": RUN_ID,
        "ingerido_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })
    log.info(
        "[BRONZE] %-22s | %6d linhas | %2d colunas | %9s | %s",
        entidade, len(dados), len(dados.columns), formatar_bytes(bytes_gravados), destino,
    )
    return dados


def ler_bronze(entidade):
    """Lê uma entidade da Bronze (o Parquet particionado é lido como um único dataset)."""
    return ler_parquet(BRONZE / entidade)


print("Motor da camada Bronze pronto.")

In [ ]:
# ============================================================
# ETAPA 2.2 — INGESTÃO BATCH: CSVs DA BASE DOS DADOS
# ============================================================
# Os 5 CSVs baixados da Base dos Dados. A leitura preserva os tipos originais:
# ajuste de tipo é trabalho da Silver.

bronze = {}

fontes_csv = [f for f in CATALOGO_FONTES if f["arquivo"]]

for spec in fontes_csv:
    entidade = spec["entidade"]
    caminho  = LANDING / spec["arquivo"]

    with monitor.etapa(f"bronze:{entidade}", camada="bronze") as metricas:
        if not caminho.exists():
            raise FileNotFoundError(
                f"Arquivo de landing ausente: {caminho}. "
                "Baixe o dataset em basedosdados.org e coloque-o em data/landing/."
            )
        bruto = pd.read_csv(caminho)
        metricas["linhas_entrada"] = len(bruto)

        bronze[entidade] = gravar_bronze(
            bruto,
            entidade=entidade,
            origem=f"basedosdados::{spec['arquivo']}",
            formato_origem="csv",
        )
        metricas["linhas_saida"]  = len(bronze[entidade])
        metricas["bytes_gerados"] = tamanho_diretorio(BRONZE / entidade)

print()
resumo_csv = pd.DataFrame([
    {"entidade": e, "linhas": len(d), "colunas": len([c for c in d.columns if not c.startswith("_")])}
    for e, d in bronze.items()
])
display(resumo_csv)

---
### 2.3 — Ingestão via API: territórios do IBGE

As tabelas do Inep identificam o território só pelo código IBGE de 7 dígitos. Sem nome de município,
UF e região, a camada Gold fica ilegível para um gestor e não dá para fazer recorte regional — que é
onde a desigualdade aparece.

Usamos a API pública de Localidades do IBGE: `/estados` (27 UFs com região) e `/municipios`
(5.571 municípios com micro e mesorregião). É o enriquecimento com fonte externa sugerido no desafio.

A função de ingestão faz *retry* com espera crescente e guarda um cache do JSON. Se a API cair, a
pipeline segue com o último payload válido e o monitor registra o alerta.

In [ ]:
# ============================================================
# ETAPA 2.3 — INGESTÃO DA API DE LOCALIDADES DO IBGE
# ============================================================

API_IBGE = "https://servicodados.ibge.gov.br/api/v1/localidades"
CACHE_DIR = LANDING / "_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def ingerir_api_json(url, nome_cache, tentativas=3, timeout=60):
    """Faz o GET, tenta de novo em caso de falha e guarda uma cópia local.

    Se a API cair e houver cache, a pipeline segue com o cache.
    """
    cache = CACHE_DIR / f"{nome_cache}.json"

    for tentativa in range(1, tentativas + 1):
        try:
            log.info("[INGESTAO] GET %s (tentativa %d/%d)", url, tentativa, tentativas)
            resposta = requests.get(url, timeout=timeout)
            resposta.raise_for_status()
            payload = resposta.json()
            cache.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
            log.info("[INGESTAO] %d registros recebidos | HTTP %d", len(payload), resposta.status_code)
            return payload
        except Exception as exc:
            espera = 2 ** tentativa
            log.warning("[INGESTAO] Falha (%s). Nova tentativa em %ds...", exc, espera)
            if tentativa < tentativas:
                time.sleep(espera)

    if cache.exists():
        monitor.registrar_alerta(
            "WARNING",
            f"API IBGE indisponível para '{nome_cache}' — usando cache local de {cache.stat().st_mtime}",
        )
        return json.loads(cache.read_text(encoding="utf-8"))

    raise RuntimeError(f"API IBGE indisponível e sem cache local para '{nome_cache}'.")


# ─── UFs ───
with monitor.etapa("bronze:dim_uf_ibge", camada="bronze") as metricas:
    payload_uf = ingerir_api_json(f"{API_IBGE}/estados", "ibge_estados")
    metricas["linhas_entrada"] = len(payload_uf)
    bronze["dim_uf_ibge"] = gravar_bronze(
        pd.json_normalize(payload_uf),
        entidade="dim_uf_ibge",
        origem=f"{API_IBGE}/estados",
        formato_origem="json",
    )
    metricas["linhas_saida"] = len(bronze["dim_uf_ibge"])

# ─── Municípios ───
with monitor.etapa("bronze:dim_municipio_ibge", camada="bronze") as metricas:
    payload_mun = ingerir_api_json(f"{API_IBGE}/municipios", "ibge_municipios")
    metricas["linhas_entrada"] = len(payload_mun)
    bronze["dim_municipio_ibge"] = gravar_bronze(
        pd.json_normalize(payload_mun),
        entidade="dim_municipio_ibge",
        origem=f"{API_IBGE}/municipios",
        formato_origem="json",
    )
    metricas["linhas_saida"] = len(bronze["dim_municipio_ibge"])

print()
display(bronze["dim_municipio_ibge"].head(3))

---
### 2.4 — Microdados de aluno

A base de alunos é a única do catálogo que passa do limite de download gratuito do portal da Base dos
Dados: são 3.867.999 linhas, 256 MB descomprimidos. A plataforma também publica os datasets no
BigQuery, e foi de lá que ela veio.

A extração está em `src/ingestion/extrair_bigquery.py` e roda no BigQuery Sandbox, que é gratuito e
não exige cartão de crédito. O script aplica duas práticas de FinOps: projeção explícita de colunas
(nunca `SELECT *`) e filtro na coluna `ano`, porque o BigQuery cobra por bytes lidos. Ele também faz
um *dry run* antes, que estima o custo sem executar.

O resultado fica em `data/landing/alunos.parquet` — 53 MB, contra 214 MB do CSV original. Esse
arquivo não vai para o Git por causa do tamanho; o passo a passo para gerá-lo está em
`docs/guia_extracao_bigquery.md`.

Se o arquivo não existir, o notebook tenta consultar o BigQuery direto (basta definir
`GCP_BILLING_PROJECT`). Não havendo nenhum dos dois, ele para com a instrução do que fazer.

In [ ]:
# ============================================================
# ETAPA 2.4 — MICRODADOS DE ALUNO
# ============================================================
# São 3,87 milhões de linhas do Inep, no grão de aluno. A tabela não cabe no
# limite de download gratuito do portal da Base dos Dados, então foi extraída do
# BigQuery público e salva em Parquet (o CSV original tem 214 MB; em Parquet
# fica com 53 MB).

ARQUIVO_ALUNOS = LANDING / "alunos.parquet"

GCP_BILLING_PROJECT = os.environ.get("GCP_BILLING_PROJECT", "")
TABELA_ALUNO_BQ = "basedosdados.br_inep_avaliacao_alfabetizacao.alunos"

# Projeção de colunas e filtro por ano: o BigQuery cobra por byte lido.
QUERY_ALUNO = f"""
    SELECT ano, id_municipio, id_escola, id_aluno, rede, serie,
           presenca, preenchimento_caderno, alfabetizado, proficiencia, peso_aluno
    FROM `{TABELA_ALUNO_BQ}`
    WHERE ano >= 2023
"""


def ingerir_alunos_arquivo():
    """Lê os microdados do arquivo local."""
    if not ARQUIVO_ALUNOS.exists():
        raise FileNotFoundError(f"{ARQUIVO_ALUNOS} não encontrado")
    log.info("[INGESTAO] Lendo microdados de %s", ARQUIVO_ALUNOS.name)
    return pd.read_parquet(ARQUIVO_ALUNOS)


def ingerir_alunos_bigquery():
    """Consulta os microdados direto no BigQuery público."""
    if not GCP_BILLING_PROJECT:
        raise RuntimeError("GCP_BILLING_PROJECT não configurado")
    import basedosdados as bd
    log.info("[INGESTAO] Consultando %s via BigQuery", TABELA_ALUNO_BQ)
    return bd.read_sql(QUERY_ALUNO, billing_project_id=GCP_BILLING_PROJECT)


# Tenta o arquivo local primeiro; se não existir, tenta o BigQuery.
with monitor.etapa("bronze:aluno", camada="bronze", sla_segundos=300) as metricas:
    try:
        df_alunos_bruto = ingerir_alunos_arquivo()
        origem_desc = f"inep/basedosdados::alunos ({ARQUIVO_ALUNOS.name})"
    except FileNotFoundError as erro_arquivo:
        log.info("[INGESTAO] Arquivo ausente (%s). Tentando BigQuery.", erro_arquivo)
        try:
            df_alunos_bruto = ingerir_alunos_bigquery()
            origem_desc = f"bigquery::{TABELA_ALUNO_BQ}"
        except Exception as erro_bq:
            raise RuntimeError(
                "Os microdados de aluno não foram encontrados.\n"
                f"  1) Coloque o arquivo em {ARQUIVO_ALUNOS}, ou\n"
                "  2) defina GCP_BILLING_PROJECT e rode:\n"
                "     python -m src.ingestion.extrair_bigquery --projeto SEU-PROJETO\n"
                f"  (erro do BigQuery: {erro_bq})"
            ) from erro_bq

    metricas["linhas_entrada"] = len(df_alunos_bruto)
    bronze["aluno"] = gravar_bronze(
        df_alunos_bruto, entidade="aluno", origem=origem_desc, formato_origem="parquet")
    metricas["linhas_saida"] = len(bronze["aluno"])
    metricas["bytes_gerados"] = tamanho_diretorio(BRONZE / "aluno")

print(f"\nMicrodados de aluno: {len(bronze['aluno']):,} linhas".replace(",", "."))
display(bronze["aluno"].head(5))

In [ ]:
# ============================================================
# ETAPA 2.5 — MANIFESTO DA CAMADA BRONZE
# ============================================================
# O manifesto registra o que entrou, de onde, quando e quanto ocupa.
# Em produção, é o que alimenta o AWS Glue Data Catalog.

caminho_manifesto = BRONZE / f"_manifesto_{RUN_ID}.json"
caminho_manifesto.write_text(
    json.dumps({"run_id": RUN_ID, "gerado_em": INGESTION_DATE, "entidades": MANIFESTO_BRONZE},
               indent=2, ensure_ascii=False),
    encoding="utf-8",
)

df_manifesto = pd.DataFrame(MANIFESTO_BRONZE)
df_manifesto["tamanho"] = df_manifesto["bytes"].apply(formatar_bytes)

print(f"BRONZE — {len(df_manifesto)} entidades | "
      f"{df_manifesto['linhas'].sum():,} linhas | "
      f"{formatar_bytes(df_manifesto['bytes'].sum())}".replace(",", "."))
print(f"Manifesto: {caminho_manifesto}\n")
display(df_manifesto[["entidade", "origem", "formato_origem", "linhas", "colunas", "tamanho"]])

---
## Etapa 3 — Qualidade de dados

O desafio pede validação de duplicidade, valores ausentes, chaves de relacionamento e consistência
entre tabelas. Em vez de espalhar `assert` pelo notebook, montamos um framework em
`src/quality/data_quality.py` com verificações encadeáveis: `check_row_count`, `check_not_null`,
`check_unique`, `check_chave_composta`, `check_domain`, `check_range`,
`check_referential_integrity`, `check_soma_igual`, `check_cross_table` e `check_freshness`.

Cada verificação tem severidade: `ERROR` interrompe a pipeline, `WARNING` só registra. Nem toda
imperfeição merece parar a esteira — os níveis de proficiência ausentes em 2023, por exemplo, são
característica conhecida da fonte, não erro de ingestão. O resultado sai em JSON por tabela, em
`data/lake/quality/`.

In [ ]:
%%writefile src/quality/data_quality.py
"""Verificacoes de qualidade dos dados.

Uso:
    dq = DataQualityCheck(df, "indicador_municipio", "bronze")
    dq.check_row_count(1000).check_not_null(["ano"]).relatorio()

Severidade ERROR conta como falha e faz o aprovar_ou_falhar() parar a pipeline.
WARNING so aparece no relatorio.
"""
import json
import logging
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None

log = logging.getLogger("pipeline.quality")


class DataQualityCheck:

    def __init__(self, df, entidade, camada, output_dir="data/lake/quality"):
        self.df = df
        self.entidade = entidade
        self.camada = camada
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.resultados = []
        self.passou = 0
        self.falhou = 0
        self.avisos = 0

    def _registrar(self, verificacao, dimensao, ok, detalhe, severidade="ERROR"):
        if ok:
            status = "PASS"
            self.passou += 1
        elif severidade == "WARNING":
            status = "WARN"
            self.avisos += 1
        else:
            status = "FAIL"
            self.falhou += 1
        self.resultados.append({"verificacao": verificacao, "dimensao": dimensao,
                                "status": status, "detalhe": detalhe,
                                "severidade": severidade})
        return self

    def _tem_coluna(self, coluna, verificacao, dimensao, severidade):
        if coluna in self.df.columns:
            return True
        self._registrar(verificacao, dimensao, False, "coluna inexistente", severidade)
        return False

    # --- completude ---
    def check_row_count(self, minimo, severidade="ERROR"):
        total = len(self.df)
        return self._registrar("row_count", "COMPLETUDE", total >= minimo,
                               f"{total} linhas (minimo esperado: {minimo})", severidade)

    def check_not_null(self, colunas, severidade="ERROR"):
        for coluna in colunas:
            if not self._tem_coluna(coluna, f"not_null:{coluna}", "COMPLETUDE", severidade):
                continue
            nulos = int(self.df[coluna].isna().sum())
            pct = nulos / len(self.df) * 100 if len(self.df) else 0
            self._registrar(f"not_null:{coluna}", "COMPLETUDE", nulos == 0,
                            f"{nulos} nulos ({pct:.2f}%)", severidade)
        return self

    # --- unicidade ---
    def check_unique(self, colunas, severidade="ERROR"):
        for coluna in colunas:
            if not self._tem_coluna(coluna, f"unique:{coluna}", "UNICIDADE", severidade):
                continue
            duplicadas = int(self.df[coluna].duplicated().sum())
            self._registrar(f"unique:{coluna}", "UNICIDADE", duplicadas == 0,
                            f"{duplicadas} duplicatas", severidade)
        return self

    def check_chave_composta(self, colunas, severidade="ERROR"):
        faltantes = [c for c in colunas if c not in self.df.columns]
        if faltantes:
            return self._registrar("chave_composta", "UNICIDADE", False,
                                   f"colunas ausentes: {faltantes}", severidade)
        duplicadas = int(self.df.duplicated(subset=colunas).sum())
        return self._registrar(f"chave_composta:{'+'.join(colunas)}", "UNICIDADE",
                               duplicadas == 0, f"{duplicadas} linhas duplicadas no grao",
                               severidade)

    # --- validade ---
    def check_domain(self, coluna, valores_validos, severidade="ERROR"):
        if not self._tem_coluna(coluna, f"dominio:{coluna}", "VALIDADE", severidade):
            return self
        fora = self.df.loc[~self.df[coluna].isin(valores_validos) & self.df[coluna].notna(), coluna]
        exemplos = sorted(pd.unique(fora))[:5]
        detalhe = f"{len(fora)} valores fora do dominio {list(valores_validos)}"
        if exemplos:
            detalhe += f" | ex.: {exemplos}"
        return self._registrar(f"dominio:{coluna}", "VALIDADE", len(fora) == 0, detalhe, severidade)

    def check_range(self, coluna, minimo=None, maximo=None, severidade="ERROR"):
        if not self._tem_coluna(coluna, f"faixa:{coluna}", "VALIDADE", severidade):
            return self
        serie = pd.to_numeric(self.df[coluna], errors="coerce").dropna()
        fora = 0
        if minimo is not None:
            fora += int((serie < minimo).sum())
        if maximo is not None:
            fora += int((serie > maximo).sum())
        return self._registrar(f"faixa:{coluna}", "VALIDADE", fora == 0,
                               f"{fora} valores fora de [{minimo}, {maximo}]", severidade)

    # --- consistencia ---
    def check_referential_integrity(self, coluna, df_referencia, coluna_referencia,
                                    severidade="ERROR"):
        if not self._tem_coluna(coluna, f"fk:{coluna}", "CONSISTENCIA", severidade):
            return self
        validos = set(df_referencia[coluna_referencia].dropna().unique())
        presentes = self.df[coluna].dropna()
        orfaos = presentes[~presentes.isin(validos)]
        detalhe = f"{len(orfaos)} registros orfaos"
        exemplos = sorted(pd.unique(orfaos))[:5]
        if exemplos:
            detalhe += f" | ex.: {exemplos}"
        return self._registrar(f"fk:{coluna}->{coluna_referencia}", "CONSISTENCIA",
                               len(orfaos) == 0, detalhe, severidade)

    def check_soma_igual(self, colunas, valor_esperado, tolerancia=0.5, severidade="WARNING"):
        faltantes = [c for c in colunas if c not in self.df.columns]
        if faltantes:
            return self._registrar("soma_colunas", "CONSISTENCIA", False,
                                   f"colunas ausentes: {faltantes}", severidade)
        subset = self.df[colunas].dropna(how="all")
        if subset.empty:
            return self._registrar(f"soma:{len(colunas)}col", "CONSISTENCIA", True,
                                   "nenhuma linha preenchida", "WARNING")
        desvio = (subset.sum(axis=1) - valor_esperado).abs()
        fora = int((desvio > tolerancia).sum())
        return self._registrar(f"soma:{len(colunas)}col={valor_esperado}", "CONSISTENCIA",
                               fora == 0,
                               f"{fora}/{len(subset)} linhas fora da tolerancia +-{tolerancia}",
                               severidade)

    def check_cross_table(self, df_referencia, chaves, coluna, coluna_referencia,
                          tolerancia=0.01, severidade="ERROR"):
        # compara o mesmo indicador vindo de duas tabelas diferentes
        faltantes = [c for c in chaves if c not in self.df.columns
                     or c not in df_referencia.columns]
        if faltantes:
            return self._registrar("cross_table", "CONSISTENCIA", False,
                                   f"chaves ausentes: {faltantes}", severidade)
        juncao = (self.df[chaves + [coluna]]
                  .merge(df_referencia[chaves + [coluna_referencia]], on=chaves, how="inner")
                  .dropna(subset=[coluna, coluna_referencia]))
        if juncao.empty:
            return self._registrar(f"cross_table:{coluna}", "CONSISTENCIA", False,
                                   "nenhuma linha em comum", severidade)
        divergentes = int(((juncao[coluna] - juncao[coluna_referencia]).abs() > tolerancia).sum())
        pct = divergentes / len(juncao) * 100
        return self._registrar(
            f"cross_table:{coluna}~{coluna_referencia}", "CONSISTENCIA", divergentes == 0,
            f"{divergentes}/{len(juncao)} divergencias ({pct:.2f}%) acima de +-{tolerancia}",
            severidade)

    def check_freshness(self, coluna="_ingestion_date", dias_maximos=1, severidade="WARNING"):
        if not self._tem_coluna(coluna, f"freshness:{coluna}", "ATUALIDADE", severidade):
            return self
        datas = pd.to_datetime(self.df[coluna], errors="coerce").dropna()
        if datas.empty:
            return self._registrar(f"freshness:{coluna}", "ATUALIDADE", False,
                                   "sem datas validas", severidade)
        hoje = pd.Timestamp(datetime.now(timezone.utc).date())
        atraso = (hoje - datas.max().normalize()).days
        return self._registrar(f"freshness:{coluna}", "ATUALIDADE", atraso <= dias_maximos,
                               f"ingestao mais recente ha {atraso} dia(s)", severidade)

    def check_expectativa(self, nome, condicao_ok, detalhe, dimensao="REGRA_NEGOCIO",
                          severidade="ERROR"):
        # para regras especificas que nao cabem nos checks acima
        return self._registrar(nome, dimensao, bool(condicao_ok), detalhe, severidade)

    # --- saida ---
    @property
    def score(self):
        total = self.passou + self.falhou + self.avisos
        return round(self.passou / total * 100, 1) if total else 0.0

    def relatorio(self, salvar=True, imprimir=True):
        linhas = [(r["status"], r["dimensao"], r["verificacao"], r["detalhe"])
                  for r in self.resultados]
        if imprimir:
            print("\n" + "=" * 96)
            print(f"  QUALIDADE - {self.entidade.upper()} [{self.camada.upper()}]")
            print("=" * 96)
            if tabulate:
                print(tabulate(linhas, headers=["Status", "Dimensao", "Verificacao", "Detalhe"],
                               tablefmt="rounded_outline"))
            else:
                print(pd.DataFrame(linhas, columns=["Status", "Dimensao", "Verificacao",
                                                    "Detalhe"]).to_string(index=False))
            print(f"  Score: {self.score}%  |  PASS {self.passou}  FAIL {self.falhou}  "
                  f"WARN {self.avisos}")

        dados = {"entidade": self.entidade, "camada": self.camada, "score": self.score,
                 "linhas_avaliadas": len(self.df),
                 "gerado_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                 "resumo": {"pass": self.passou, "fail": self.falhou, "warn": self.avisos},
                 "verificacoes": self.resultados}
        if salvar:
            destino = self.output_dir / f"dq_{self.camada}_{self.entidade}.json"
            destino.write_text(json.dumps(dados, indent=2, ensure_ascii=False), encoding="utf-8")
            dados["arquivo"] = str(destino)

        log.info("[DQ:%s] %s | score=%s%% | PASS=%d FAIL=%d WARN=%d", self.camada.upper(),
                 self.entidade, self.score, self.passou, self.falhou, self.avisos)
        return dados

    def aprovar_ou_falhar(self):
        # se tiver falha critica, para a pipeline aqui
        if self.falhou:
            criticas = [r for r in self.resultados if r["status"] == "FAIL"]
            detalhes = "; ".join(f"{r['verificacao']} ({r['detalhe']})" for r in criticas)
            raise ValueError(f"[QUALITY GATE] {self.entidade} [{self.camada}]: "
                             f"{self.falhou} verificacao(oes) critica(s) falharam -> {detalhes}")
        return self


def consolidar_relatorios(relatorios):
    """Junta varios relatorios num painel unico."""
    return pd.DataFrame([{
        "camada": r["camada"],
        "entidade": r["entidade"],
        "linhas": r["linhas_avaliadas"],
        "score_%": r["score"],
        "pass": r["resumo"]["pass"],
        "fail": r["resumo"]["fail"],
        "warn": r["resumo"]["warn"],
    } for r in relatorios]).sort_values(["camada", "entidade"]).reset_index(drop=True)

In [ ]:
# ============================================================
# ETAPA 3.2 — QUALITY GATE DA CAMADA BRONZE
# ============================================================
from src.quality.data_quality import DataQualityCheck, consolidar_relatorios

COLUNAS_NIVEL = [f"proporcao_aluno_nivel_{i}" for i in range(9)]
UFS_VALIDAS   = set(bronze["dim_uf_ibge"]["sigla"])

# Os níveis de proficiência só existem a partir de 2024. Em vez de acusar 9
# colunas nulas, a verificação olha a cobertura no ano em que elas devem existir.
_ind_mun = bronze["indicador_municipio"]
ANO_MAIS_RECENTE = int(_ind_mun["ano"].max())
COBERTURA_NIVEIS = _ind_mun.loc[_ind_mun["ano"] == ANO_MAIS_RECENTE, COLUNAS_NIVEL[0]].notna().mean()

# As regras de cada tabela, declaradas em uma lista.
REGRAS_BRONZE = {
    "indicador_municipio": lambda dq: (
        dq.check_row_count(10_000)
          .check_not_null(["ano", "id_municipio", "serie", "rede", "taxa_alfabetizacao"])
          .check_chave_composta(["ano", "id_municipio", "serie", "rede"])
          .check_domain("rede", DIM_REDE.keys())
          .check_domain("serie", [SERIE_AVALIADA])
          .check_range("taxa_alfabetizacao", 0, 100)
          .check_range("media_portugues", 400, 1000)
          .check_expectativa(
              "niveis_saeb:cobertura_ano_recente",
              COBERTURA_NIVEIS > 0.99,
              f"{COBERTURA_NIVEIS:.1%} das linhas de {ANO_MAIS_RECENTE} têm distribuição por "
              f"nível de proficiência (a fonte não publica níveis para anos anteriores)",
              dimensao="COMPLETUDE", severidade="WARNING")
          .check_soma_igual(COLUNAS_NIVEL, 100, tolerancia=0.5, severidade="WARNING")
          .check_freshness()
    ),
    "indicador_uf": lambda dq: (
        dq.check_row_count(100)
          .check_not_null(["ano", "sigla_uf", "serie", "rede", "taxa_alfabetizacao"])
          .check_chave_composta(["ano", "sigla_uf", "serie", "rede"])
          .check_domain("rede", DIM_REDE.keys())
          .check_domain("sigla_uf", UFS_VALIDAS)
          .check_range("taxa_alfabetizacao", 0, 100)
    ),
    "meta_municipio": lambda dq: (
        dq.check_row_count(10_000)
          .check_not_null(["ano", "id_municipio", "rede"])
          .check_chave_composta(["ano", "id_municipio", "rede"])
          .check_range("taxa_alfabetizacao", 0, 100)
          .check_range("meta_alfabetizacao_2030", 0, 100)
          .check_not_null(["taxa_alfabetizacao"], severidade="WARNING")
    ),
    "meta_uf": lambda dq: (
        dq.check_row_count(50)
          .check_not_null(["ano", "sigla_uf", "rede"])
          .check_chave_composta(["ano", "sigla_uf", "rede"])
          .check_domain("sigla_uf", UFS_VALIDAS)
          .check_not_null(["taxa_alfabetizacao"], severidade="WARNING")
    ),
    "meta_brasil": lambda dq: (
        dq.check_row_count(3)
          .check_not_null(["ano", "rede", "taxa_alfabetizacao"])
          .check_chave_composta(["ano", "rede"])
          .check_range("taxa_alfabetizacao", 0, 100)
          .check_range("meta_alfabetizacao_2030", 0, 100)
    ),
    "dim_uf_ibge": lambda dq: (
        dq.check_row_count(27)
          .check_unique(["id", "sigla"])
          .check_not_null(["id", "sigla", "nome", "regiao.nome"])
    ),
    "dim_municipio_ibge": lambda dq: (
        dq.check_row_count(5_500)
          .check_unique(["id"])
          .check_not_null(["id", "nome"])
    ),
    # O mesmo id_aluno aparece nos dois anos, então a chave é (ano, id_aluno).
    # Proficiência nula indica aluno ausente, e o registro precisa existir.
    "aluno": lambda dq: (
        dq.check_row_count(1_000)
          .check_chave_composta(["ano", "id_aluno"])
          .check_not_null(["id_aluno", "ano", "id_municipio", "rede"])
          .check_domain("rede", DIM_REDE.keys())
          .check_domain("serie", [SERIE_AVALIADA])
          .check_range("proficiencia", 200, 1000)
          .check_range("peso_aluno", 0, 1000)
          .check_not_null(["proficiencia"], severidade="WARNING")
          .check_referential_integrity("id_municipio", bronze["dim_municipio_ibge"], "id")
    ),
}

relatorios_bronze = []
with monitor.etapa("quality_gate:bronze", camada="bronze") as metricas:
    for entidade, aplicar_regras in REGRAS_BRONZE.items():
        dq = DataQualityCheck(bronze[entidade], entidade, "bronze", output_dir=QUALITY)
        aplicar_regras(dq)
        relatorios_bronze.append(dq.relatorio(imprimir=(entidade == "indicador_municipio")))
        dq.aprovar_ou_falhar()          # falha crítica interrompe a pipeline
        metricas["linhas_entrada"] += len(bronze[entidade])

print("\nPAINEL DE QUALIDADE — CAMADA BRONZE")
display(consolidar_relatorios(relatorios_bronze))

---
## Etapa 4 — Camada SILVER

Aqui o dado deixa de ser arquivo e vira modelo: padronização de nomes e tipos (`ano` → `int16`,
`id_municipio` → `int32`), normalização de chaves (`id_municipio` validado como código IBGE de 7
dígitos, `id_uf` derivado dos 2 primeiros), decodificação de domínio (`rede = 3` → `"Municipal"`),
tratamento de nulos (os estruturais ficam, os indevidos viram linha rejeitada), deduplicação pelo
grão do catálogo, normalização de estrutura (as 9 colunas `proporcao_aluno_nivel_*` viram tabela
longa) e, por fim, a integração das bases.

```
   dim_uf ────────┐
   dim_municipio ─┤
   dim_meta_municipio ──┼──►  fato_alfabetizacao_municipio   (ano × município × rede)
   fato_indicador_municipio ─┤
   fato_indicador_uf ────────┘

   fato_distribuicao_nivel   (ano × município × rede × nível)
   fato_aluno                (grão de aluno)
```

A Silver nunca é escrita por cima da Bronze: toda tabela aqui é reconstruível a partir dela, então
erro de regra de negócio se corrige por reprocessamento.

In [ ]:
# ============================================================
# ETAPA 4.1 — MOTOR DE GRAVAÇÃO DA SILVER + DIMENSÕES TERRITORIAIS
# ============================================================
silver = {}
INVENTARIO_SILVER = []


def gravar_silver(df, tabela, particoes=None):
    """Escreve uma tabela Silver em Parquet, opcionalmente particionada."""
    dados = df.copy()
    dados["_silver_timestamp"] = RUN_ID
    dados["_pipeline_run_id"]  = RUN_ID

    destino = SILVER / tabela
    if destino.exists():
        shutil.rmtree(destino)
    destino.mkdir(parents=True, exist_ok=True)

    if particoes:
        dados = preparar_particoes(dados, particoes)
        dados.to_parquet(destino, engine="pyarrow", compression="snappy",
                         partition_cols=particoes, index=False)
    else:
        dados.to_parquet(destino / "part-00000.parquet", engine="pyarrow",
                         compression="snappy", index=False)

    INVENTARIO_SILVER.append({
        "tabela": tabela,
        "linhas": len(dados),
        "colunas": len(dados.columns),
        "particionada_por": ",".join(particoes) if particoes else "-",
        "bytes": tamanho_diretorio(destino),
        "caminho_s3": f"{S3_PREFIX['silver']}/{tabela}/",
    })
    log.info("[SILVER] %-32s | %6d linhas | %2d colunas | %s",
             tabela, len(dados), len(dados.columns), formatar_bytes(tamanho_diretorio(destino)))
    silver[tabela] = dados
    return dados


# ─── dim_uf: as 27 UFs com região ───
with monitor.etapa("silver:dim_uf", camada="silver") as metricas:
    origem_uf = bronze["dim_uf_ibge"]
    metricas["linhas_entrada"] = len(origem_uf)

    dim_uf = pd.DataFrame({
        "id_uf":       origem_uf["id"].astype("int16"),
        "sigla_uf":    origem_uf["sigla"].str.strip().str.upper(),
        "nome_uf":     origem_uf["nome"].str.strip(),
        "sigla_regiao": origem_uf["regiao.sigla"].str.strip(),
        "nome_regiao":  origem_uf["regiao.nome"].str.strip(),
    }).drop_duplicates(subset=["id_uf"]).sort_values("sigla_uf").reset_index(drop=True)

    gravar_silver(dim_uf, "dim_uf")
    metricas["linhas_saida"] = len(dim_uf)


# ─── dim_municipio: 5,5 mil municípios ───
with monitor.etapa("silver:dim_municipio", camada="silver") as metricas:
    origem_mun = bronze["dim_municipio_ibge"]
    metricas["linhas_entrada"] = len(origem_mun)

    def coluna_opcional(df, nome, padrao=None):
        """A API do IBGE tem municípios sem microrregião — não podemos assumir o campo."""
        return df[nome] if nome in df.columns else pd.Series(padrao, index=df.index)

    dim_municipio = pd.DataFrame({
        "id_municipio":      origem_mun["id"].astype("int32"),
        "nome_municipio":    origem_mun["nome"].str.strip(),
        "nome_microrregiao": coluna_opcional(origem_mun, "microrregiao.nome"),
        "nome_mesorregiao":  coluna_opcional(origem_mun, "microrregiao.mesorregiao.nome"),
        "nome_regiao_imediata": coluna_opcional(origem_mun, "regiao-imediata.nome"),
    })

    # Os 2 primeiros dígitos do código IBGE são a UF. Vem daqui e não do JSON
    # aninhado, porque existe município sem microrregião preenchida.
    dim_municipio["id_uf"] = (dim_municipio["id_municipio"] // 100_000).astype("int16")
    dim_municipio = dim_municipio.merge(dim_uf, on="id_uf", how="left")

    # Linhas sem UF resolvida são rejeitadas (não podem participar de nenhuma análise regional)
    rejeitadas = dim_municipio["sigla_uf"].isna().sum()
    dim_municipio = dim_municipio[dim_municipio["sigla_uf"].notna()].reset_index(drop=True)
    metricas["linhas_rejeitadas"] = int(rejeitadas)

    gravar_silver(dim_municipio, "dim_municipio")
    metricas["linhas_saida"] = len(dim_municipio)

display(dim_municipio.head(3))

In [ ]:
# ============================================================
# ETAPA 4.2 — FATOS DO INDICADOR (MUNICÍPIO E UF)
# ============================================================

def padronizar_indicador(df, chave_territorial, chave_grao):
    """Limpeza, tipagem e deduplicação de uma tabela de indicador.

    Devolve o dataframe limpo e a quantidade de linhas rejeitadas. Esse número
    alimenta o monitor, que alerta se passar de 5%.
    """
    dados = df.copy()
    total_entrada = len(dados)

    # 1) Tipagem explícita, senão 'ano' vira float por causa de um nulo
    dados["ano"]   = pd.to_numeric(dados["ano"], errors="coerce").astype("Int16")
    dados["serie"] = pd.to_numeric(dados["serie"], errors="coerce").astype("Int8")
    dados["rede"]  = pd.to_numeric(dados["rede"], errors="coerce").astype("Int8")
    for coluna in ["taxa_alfabetizacao", "media_portugues"]:
        dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce").round(2)

    # 2) O que não passar destas regras não entra na Silver
    valido = (
        dados["ano"].notna()
        & dados["rede"].isin(list(DIM_REDE.keys()))
        & dados["taxa_alfabetizacao"].between(0, 100)
    )
    if chave_territorial == "id_municipio":
        dados["id_municipio"] = pd.to_numeric(dados["id_municipio"], errors="coerce").astype("Int32")
        valido &= dados["id_municipio"].between(1_000_000, 9_999_999)   # código IBGE de 7 dígitos
    else:
        dados["sigla_uf"] = dados["sigla_uf"].str.strip().str.upper()
        valido &= dados["sigla_uf"].isin(set(dim_uf["sigla_uf"]))

    rejeitadas = int((~valido).sum())
    dados = dados[valido].copy()

    # 3) Troca o código da rede pelo nome correspondente
    dados["rede_nome"] = dados["rede"].map(DIM_REDE)

    # 4) Remove duplicatas, mantendo a ingestão mais recente
    dados = (dados.sort_values("_ingestion_timestamp")
                  .drop_duplicates(subset=chave_grao, keep="last")
                  .reset_index(drop=True))

    log.info("[SILVER] padronizacao %-18s | entrada=%d | rejeitadas=%d | saida=%d",
             chave_territorial, total_entrada, rejeitadas, len(dados))
    return dados, rejeitadas


COLUNAS_FATO_BASE = ["ano", "serie", "rede", "rede_nome", "taxa_alfabetizacao", "media_portugues"]

# ─── fato_indicador_municipio ───
with monitor.etapa("silver:fato_indicador_municipio", camada="silver") as metricas:
    metricas["linhas_entrada"] = len(bronze["indicador_municipio"])
    limpo, rejeitadas = padronizar_indicador(
        bronze["indicador_municipio"], "id_municipio", ["ano", "id_municipio", "serie", "rede"])
    metricas["linhas_rejeitadas"] = rejeitadas

    fato_indicador_municipio = limpo[["id_municipio"] + COLUNAS_FATO_BASE].copy()
    fato_indicador_municipio["id_uf"] = (
        fato_indicador_municipio["id_municipio"] // 100_000).astype("Int16")
    fato_indicador_municipio["alcancou_meta_2030"] = (
        fato_indicador_municipio["taxa_alfabetizacao"] >= META_NACIONAL_2030)

    gravar_silver(fato_indicador_municipio, "fato_indicador_municipio", particoes=["ano"])
    metricas["linhas_saida"] = len(fato_indicador_municipio)

# ─── fato_indicador_uf ───
with monitor.etapa("silver:fato_indicador_uf", camada="silver") as metricas:
    metricas["linhas_entrada"] = len(bronze["indicador_uf"])
    limpo_uf, rejeitadas_uf = padronizar_indicador(
        bronze["indicador_uf"], "sigla_uf", ["ano", "sigla_uf", "serie", "rede"])
    metricas["linhas_rejeitadas"] = rejeitadas_uf

    fato_indicador_uf = (limpo_uf[["sigla_uf"] + COLUNAS_FATO_BASE]
                         .merge(dim_uf[["sigla_uf", "id_uf", "nome_uf", "nome_regiao"]],
                                on="sigla_uf", how="left"))
    gravar_silver(fato_indicador_uf, "fato_indicador_uf", particoes=["ano"])
    metricas["linhas_saida"] = len(fato_indicador_uf)

# ─── fato_distribuicao_nivel (normalização das 9 colunas de proporção) ───
with monitor.etapa("silver:fato_distribuicao_nivel", camada="silver") as metricas:
    largo = limpo[["ano", "id_municipio", "rede", "rede_nome"] + COLUNAS_NIVEL]
    metricas["linhas_entrada"] = len(largo)

    longo = largo.melt(
        id_vars=["ano", "id_municipio", "rede", "rede_nome"],
        value_vars=COLUNAS_NIVEL, var_name="coluna_nivel", value_name="proporcao_alunos",
    ).dropna(subset=["proporcao_alunos"])
    longo["nivel"] = longo["coluna_nivel"].str.extract(r"(\d)$").astype("int8")
    fato_distribuicao_nivel = (longo.drop(columns=["coluna_nivel"])
                                    .sort_values(["ano", "id_municipio", "rede", "nivel"])
                                    .reset_index(drop=True))

    gravar_silver(fato_distribuicao_nivel, "fato_distribuicao_nivel", particoes=["ano"])
    metricas["linhas_saida"] = len(fato_distribuicao_nivel)

display(fato_indicador_municipio.head(3))

In [ ]:
# ============================================================
# ETAPA 4.3 — TRAJETÓRIAS DE META (BRASIL, UF E MUNICÍPIO)
# ============================================================
# As metas vêm como colunas (meta_alfabetizacao_2024, _2025... _2030), formato
# que não permite comparar com o realizado ano a ano. Aqui viram linhas.
# Detalhe: a mesma meta aparece nas linhas de 2023 e 2024, às vezes com valor
# diferente. Fica valendo a mais recente.

def normalizar_rede_texto(serie):
    return (serie.astype(str).str.strip().str.lower()
                 .map(REDE_TEXTO_PARA_CODIGO).astype("Int8"))


def trajetoria_de_metas(df, chaves_territoriais):
    """Vira as colunas meta_alfabetizacao_AAAA em linhas."""
    dados = df.copy()
    dados["rede"] = normalizar_rede_texto(dados["rede"])
    colunas_meta = [c for c in dados.columns if c.startswith("meta_alfabetizacao_")]

    longo = dados.melt(
        id_vars=chaves_territoriais + ["rede", "ano"],
        value_vars=colunas_meta, var_name="coluna_meta", value_name="meta_taxa",
    ).dropna(subset=["meta_taxa"])

    longo["ano_meta"] = longo["coluna_meta"].str.extract(r"(\d{4})$").astype("int16")
    longo["meta_taxa"] = pd.to_numeric(longo["meta_taxa"], errors="coerce").round(2)

    # fica valendo a versão declarada no ano mais recente
    vigente = (longo.sort_values("ano")
                    .drop_duplicates(subset=chaves_territoriais + ["rede", "ano_meta"], keep="last"))
    return (vigente[chaves_territoriais + ["rede", "ano_meta", "meta_taxa"]]
            .sort_values(chaves_territoriais + ["ano_meta"])
            .reset_index(drop=True))


# ─── Metas por município ───
with monitor.etapa("silver:dim_meta_municipio", camada="silver") as metricas:
    origem = bronze["meta_municipio"].copy()
    origem["id_municipio"] = pd.to_numeric(origem["id_municipio"], errors="coerce").astype("Int32")
    metricas["linhas_entrada"] = len(origem)

    dim_meta_municipio = trajetoria_de_metas(origem, ["id_municipio"])
    dim_meta_municipio["rede_nome"] = dim_meta_municipio["rede"].map(DIM_REDE)
    gravar_silver(dim_meta_municipio, "dim_meta_municipio")
    metricas["linhas_saida"] = len(dim_meta_municipio)

# ─── Metas por UF ───
with monitor.etapa("silver:dim_meta_uf", camada="silver") as metricas:
    origem_uf_meta = bronze["meta_uf"].copy()
    origem_uf_meta["sigla_uf"] = origem_uf_meta["sigla_uf"].str.strip().str.upper()
    metricas["linhas_entrada"] = len(origem_uf_meta)

    dim_meta_uf = trajetoria_de_metas(origem_uf_meta, ["sigla_uf"])
    dim_meta_uf["rede_nome"] = dim_meta_uf["rede"].map(DIM_REDE)
    gravar_silver(dim_meta_uf, "dim_meta_uf")
    metricas["linhas_saida"] = len(dim_meta_uf)

# ─── Metas nacionais ───
with monitor.etapa("silver:dim_meta_brasil", camada="silver") as metricas:
    origem_br = bronze["meta_brasil"].copy()
    origem_br["pais"] = "BR"
    metricas["linhas_entrada"] = len(origem_br)

    dim_meta_brasil = trajetoria_de_metas(origem_br, ["pais"])
    dim_meta_brasil["rede_nome"] = dim_meta_brasil["rede"].map(DIM_REDE)
    gravar_silver(dim_meta_brasil, "dim_meta_brasil")
    metricas["linhas_saida"] = len(dim_meta_brasil)

# ─── Participação e nível agregado do município (atributos exclusivos da tabela de metas) ───
with monitor.etapa("silver:fato_participacao_municipio", camada="silver") as metricas:
    participacao = bronze["meta_municipio"].copy()
    participacao["id_municipio"] = pd.to_numeric(
        participacao["id_municipio"], errors="coerce").astype("Int32")
    participacao["rede"] = normalizar_rede_texto(participacao["rede"])
    metricas["linhas_entrada"] = len(participacao)

    fato_participacao_municipio = (
        participacao[["ano", "id_municipio", "rede", "percentual_participacao",
                      "nivel_alfabetizacao", "taxa_alfabetizacao"]]
        .rename(columns={"taxa_alfabetizacao": "taxa_alfabetizacao_fonte_meta"})
        .dropna(subset=["percentual_participacao"])
        .drop_duplicates(subset=["ano", "id_municipio", "rede"])
        .reset_index(drop=True)
    )
    fato_participacao_municipio["ano"] = fato_participacao_municipio["ano"].astype("Int16")
    gravar_silver(fato_participacao_municipio, "fato_participacao_municipio", particoes=["ano"])
    metricas["linhas_saida"] = len(fato_participacao_municipio)

print("\nTrajetória de metas — exemplo (município):")
display(dim_meta_municipio.head(7))
print("Anos de meta disponíveis:", sorted(dim_meta_municipio["ano_meta"].unique()))

In [ ]:
# ============================================================
# ETAPA 4.4 — SILVER DE MICRODADOS DE ALUNO
# ============================================================
# Três características do dado real que mudam o cálculo:
#   1. nem todo aluno fez a prova (presenca / preenchimento_caderno)
#   2. peso_aluno é peso amostral, então a taxa é média ponderada e não simples
#   3. alfabetizado já vem da fonte, então é preservado e apenas conferido

with monitor.etapa("silver:fato_aluno", camada="silver", sla_segundos=300) as metricas:
    origem_aluno = bronze["aluno"].copy()
    metricas["linhas_entrada"] = len(origem_aluno)

    # a fonte chama a coluna de 'proficiencia'; aqui ela é renomeada
    if "proficiencia" in origem_aluno.columns and "proficiencia_portugues" not in origem_aluno.columns:
        origem_aluno = origem_aluno.rename(columns={"proficiencia": "proficiencia_portugues"})

    obrigatorias = {"id_aluno", "ano", "id_municipio", "proficiencia_portugues"}
    faltantes = obrigatorias - set(origem_aluno.columns)
    if faltantes:
        raise KeyError(f"Microdados sem as colunas obrigatórias: {faltantes}")

    origem_aluno["ano"]          = pd.to_numeric(origem_aluno["ano"], errors="coerce").astype("Int16")
    origem_aluno["id_municipio"] = pd.to_numeric(origem_aluno["id_municipio"], errors="coerce").astype("Int32")
    origem_aluno["rede"]         = pd.to_numeric(origem_aluno["rede"], errors="coerce").astype("Int8")
    origem_aluno["serie"]        = pd.to_numeric(origem_aluno["serie"], errors="coerce").astype("Int8")
    origem_aluno["proficiencia_portugues"] = pd.to_numeric(
        origem_aluno["proficiencia_portugues"], errors="coerce")
    origem_aluno["peso_aluno"] = pd.to_numeric(
        origem_aluno.get("peso_aluno", 1.0), errors="coerce").fillna(1.0)

    # Aluno sem prova continua válido, porque entra no denominador da
    # participação. Descartadas são as chaves inválidas e a proficiência fora
    # da faixa.
    proficiencia = origem_aluno["proficiencia_portugues"]
    valido = (
        origem_aluno["id_aluno"].notna()
        & origem_aluno["ano"].notna()
        & origem_aluno["id_municipio"].between(1_000_000, 9_999_999)
        & (proficiencia.isna() | proficiencia.between(200, 1000))
    )
    metricas["linhas_rejeitadas"] = int((~valido).sum())

    # o mesmo aluno aparece nos dois anos, então a chave é (ano, id_aluno)
    fato_aluno = (origem_aluno[valido]
                  .drop_duplicates(subset=["ano", "id_aluno"])
                  .copy())

    # ── Marca quem efetivamente fez a prova ──
    fato_aluno["avaliado"] = fato_aluno["proficiencia_portugues"].notna()

    # ── Alfabetizado = 743 pontos ou mais (regra do Inep) ──
    pela_regra = fato_aluno["proficiencia_portugues"] >= PONTO_CORTE_SAEB
    if "alfabetizado" in fato_aluno.columns:
        fato_aluno["alfabetizado"] = (
            pd.to_numeric(fato_aluno["alfabetizado"], errors="coerce").fillna(0).astype(bool))
        avaliados = fato_aluno[fato_aluno["avaliado"]]
        divergencia = (avaliados["alfabetizado"] != pela_regra[avaliados.index]).mean() * 100
        log.info("[SILVER] 'alfabetizado' preservado da fonte | divergência vs. regra dos %d "
                 "pontos: %.3f%%", PONTO_CORTE_SAEB, divergencia)
    else:
        fato_aluno["alfabetizado"] = pela_regra.fillna(False)

    fato_aluno["rede_nome"] = fato_aluno["rede"].map(DIM_REDE)
    fato_aluno["faixa_proficiencia"] = pd.cut(
        fato_aluno["proficiencia_portugues"],
        bins=[0, 650, 700, PONTO_CORTE_SAEB, 800, 1000],
        labels=["Muito baixa", "Baixa", "Abaixo do corte", "Adequada", "Avançada"],
    ).astype(str)

    colunas_saida = [c for c in [
        "id_aluno", "ano", "id_municipio", "id_escola", "rede", "rede_nome", "serie",
        "proficiencia_portugues", "alfabetizado", "avaliado", "faixa_proficiencia",
        "presenca", "preenchimento_caderno", "peso_aluno",
        # colunas exclusivas do fallback simulado
        "sexo", "idade", "tipo_localizacao", "data_avaliacao",
    ] if c in fato_aluno.columns]
    fato_aluno = fato_aluno[colunas_saida].reset_index(drop=True)

    gravar_silver(fato_aluno, "fato_aluno", particoes=["ano"])
    metricas["linhas_saida"] = len(fato_aluno)

avaliados = fato_aluno[fato_aluno["avaliado"]]
taxa_simples = avaliados["alfabetizado"].mean() * 100
taxa_ponderada = np.average(avaliados["alfabetizado"], weights=avaliados["peso_aluno"]) * 100

print(f"\nAlunos no cadastro     : {len(fato_aluno):,}".replace(",", "."))
print(f"Efetivamente avaliados : {len(avaliados):,} "
      f"({len(avaliados)/len(fato_aluno)*100:.1f}%)".replace(",", "."))
print(f"Taxa (média simples)   : {taxa_simples:.2f}%")
print(f"Taxa (média ponderada) : {taxa_ponderada:.2f}%   <- é assim que o indicador é calculado")
display(fato_aluno.head(5))

---
### 4.5 — Integração das bases

É aqui que sete tabelas viram uma visão do município. A integração precisa resolver três
incompatibilidades reais entre as fontes:

| Problema | Solução |
|---|---|
| `rede` numérica no indicador, texto nas metas | dicionário `DIM_REDE` + validação cruzada |
| metas em colunas por ano, realizado em linhas por ano | trajetória normalizada e junção `ano = ano_meta` |
| território só por código | `dim_municipio` e `dim_uf` da API do IBGE |

O resultado é `fato_alfabetizacao_municipio`, com grão `ano × município × rede` e três blocos de
colunas: identificação (município, UF, região, rede), realizado (taxa, média de proficiência,
participação) e comparação (meta do ano, gap, meta 2030, taxa da UF e do Brasil).

In [ ]:
# ============================================================
# ETAPA 4.5 — TABELA INTEGRADA: fato_alfabetizacao_municipio
# ============================================================

def com_chaves_inteiras(df, colunas):
    """Deixa as chaves todas como int64 antes do merge.

    Misturar Int32 com int32 gera merge vazio sem lançar erro.
    """
    dados = df.dropna(subset=colunas).copy()
    for coluna in colunas:
        dados[coluna] = dados[coluna].astype("int64")
    return dados


with monitor.etapa("silver:fato_alfabetizacao_municipio", camada="silver") as metricas:
    base = com_chaves_inteiras(fato_indicador_municipio, ["ano", "id_municipio", "rede"])
    metricas["linhas_entrada"] = len(base)

    # 1) Território (IBGE)
    territorio = dim_municipio[["id_municipio", "nome_municipio", "sigla_uf", "nome_uf",
                                "nome_regiao", "nome_mesorregiao"]]
    integrada = base.merge(com_chaves_inteiras(territorio, ["id_municipio"]),
                           on="id_municipio", how="left")

    # 2) Meta do próprio ano (trajetória normalizada)
    metas = com_chaves_inteiras(dim_meta_municipio, ["id_municipio", "rede", "ano_meta"])
    integrada = integrada.merge(
        metas[["id_municipio", "rede", "ano_meta", "meta_taxa"]]
            .rename(columns={"ano_meta": "ano", "meta_taxa": "meta_taxa_ano"}),
        on=["id_municipio", "rede", "ano"], how="left",
    )

    # 3) Meta final de 2030
    meta_2030 = (metas[metas["ano_meta"] == 2030][["id_municipio", "rede", "meta_taxa"]]
                 .rename(columns={"meta_taxa": "meta_taxa_2030"}))
    integrada = integrada.merge(meta_2030, on=["id_municipio", "rede"], how="left")
    integrada["meta_taxa_2030"] = integrada["meta_taxa_2030"].fillna(META_NACIONAL_2030)

    # 4) Participação na avaliação (denominador da confiabilidade do indicador)
    participacao = com_chaves_inteiras(
        fato_participacao_municipio[["ano", "id_municipio", "rede",
                                     "percentual_participacao", "nivel_alfabetizacao"]],
        ["ano", "id_municipio", "rede"])
    integrada = integrada.merge(participacao, on=["ano", "id_municipio", "rede"], how="left")

    # 5) Contexto: taxa da UF na mesma rede e taxa nacional da rede pública
    contexto_uf = (fato_indicador_uf[["ano", "sigla_uf", "rede", "taxa_alfabetizacao"]]
                   .rename(columns={"taxa_alfabetizacao": "taxa_uf"}))
    contexto_uf = com_chaves_inteiras(contexto_uf, ["ano", "rede"])
    integrada = integrada.merge(contexto_uf, on=["ano", "sigla_uf", "rede"], how="left")

    contexto_br = (bronze["meta_brasil"][["ano", "taxa_alfabetizacao"]]
                   .rename(columns={"taxa_alfabetizacao": "taxa_brasil_publica"}))
    contexto_br["ano"] = contexto_br["ano"].astype("int64")
    integrada = integrada.merge(contexto_br, on="ano", how="left")

    # 6) As contas que interessam
    integrada["gap_meta_ano"]     = (integrada["taxa_alfabetizacao"] - integrada["meta_taxa_ano"]).round(2)
    integrada["atingiu_meta_ano"] = integrada["gap_meta_ano"] >= 0
    integrada["gap_meta_2030"]    = (integrada["meta_taxa_2030"] - integrada["taxa_alfabetizacao"]).round(2)
    integrada["dif_vs_uf"]        = (integrada["taxa_alfabetizacao"] - integrada["taxa_uf"]).round(2)
    integrada["dif_vs_brasil"]    = (integrada["taxa_alfabetizacao"] - integrada["taxa_brasil_publica"]).round(2)

    fato_alfabetizacao_municipio = integrada
    gravar_silver(fato_alfabetizacao_municipio, "fato_alfabetizacao_municipio", particoes=["ano"])
    metricas["linhas_saida"] = len(fato_alfabetizacao_municipio)

# ─── Cobertura de cada join ───
# Cobertura baixa aqui é esperada: as metas só existem de 2024 em diante e só
# na rede Municipal. Fica registrado para não parecer defeito depois.
def preenchimento(coluna):
    return round(fato_alfabetizacao_municipio[coluna].notna().mean() * 100, 2)

cobertura = pd.DataFrame([
    {"junção": "território (IBGE)", "coluna": "nome_municipio",
     "preenchido_%": preenchimento("nome_municipio"), "esperado": "100% — toda linha tem município"},
    {"junção": "meta do ano", "coluna": "meta_taxa_ano",
     "preenchido_%": preenchimento("meta_taxa_ano"),
     "esperado": "~22% — metas existem só de 2024 em diante e só na rede Municipal"},
    {"junção": "meta 2030", "coluna": "meta_taxa_2030",
     "preenchido_%": preenchimento("meta_taxa_2030"),
     "esperado": "100% — sem meta municipal, aplica-se a meta nacional de 80%"},
    {"junção": "participação", "coluna": "percentual_participacao",
     "preenchido_%": preenchimento("percentual_participacao"),
     "esperado": "~44% — só a rede Municipal informa participação"},
    {"junção": "contexto UF", "coluna": "taxa_uf",
     "preenchido_%": preenchimento("taxa_uf"),
     "esperado": "~100% — indicador estadual na mesma rede"},
])
print("COBERTURA DAS JUNÇÕES\n")
display(cobertura)
display(fato_alfabetizacao_municipio.head(3))

In [ ]:
# ============================================================
# ETAPA 4.6 — QUALITY GATE DA CAMADA SILVER
# ============================================================
# Duas conferências que só fazem sentido depois da integração:
#   (a) o de-para de 'rede' está correto?
#   (b) os microdados batem com o indicador oficial?

relatorios_silver = []

with monitor.etapa("quality_gate:silver", camada="silver") as metricas:

    # ─── Dimensões ───
    dq_uf = (DataQualityCheck(dim_uf, "dim_uf", "silver", output_dir=QUALITY)
             .check_row_count(27)
             .check_unique(["id_uf", "sigla_uf"])
             .check_not_null(["id_uf", "sigla_uf", "nome_uf", "nome_regiao"]))
    relatorios_silver.append(dq_uf.relatorio(imprimir=False))
    dq_uf.aprovar_ou_falhar()

    dq_mun = (DataQualityCheck(dim_municipio, "dim_municipio", "silver", output_dir=QUALITY)
              .check_row_count(5_500)
              .check_unique(["id_municipio"])
              .check_not_null(["id_municipio", "nome_municipio", "sigla_uf", "nome_regiao"])
              .check_referential_integrity("id_uf", dim_uf, "id_uf")
              .check_not_null(["nome_microrregiao"], severidade="WARNING"))
    relatorios_silver.append(dq_mun.relatorio(imprimir=False))
    dq_mun.aprovar_ou_falhar()

    # ─── Fatos ───
    dq_fato = (DataQualityCheck(fato_indicador_municipio, "fato_indicador_municipio",
                                "silver", output_dir=QUALITY)
               .check_row_count(10_000)
               .check_chave_composta(["ano", "id_municipio", "serie", "rede"])
               .check_not_null(["ano", "id_municipio", "rede", "rede_nome", "taxa_alfabetizacao"])
               .check_domain("rede_nome", DIM_REDE.values())
               .check_range("taxa_alfabetizacao", 0, 100)
               .check_referential_integrity("id_municipio", dim_municipio, "id_municipio"))
    relatorios_silver.append(dq_fato.relatorio(imprimir=False))
    dq_fato.aprovar_ou_falhar()

    # ─── (a) Consistência entre tabelas: rede "Municipal" do indicador × tabela de metas ───
    integrada_municipal = fato_alfabetizacao_municipio[
        fato_alfabetizacao_municipio["rede"] == 3][["ano", "id_municipio", "taxa_alfabetizacao"]]
    metas_referencia = bronze["meta_municipio"][["ano", "id_municipio", "taxa_alfabetizacao"]].copy()
    metas_referencia = metas_referencia.rename(columns={"taxa_alfabetizacao": "taxa_fonte_meta"})
    metas_referencia["ano"] = metas_referencia["ano"].astype("int64")
    metas_referencia["id_municipio"] = metas_referencia["id_municipio"].astype("int64")

    dq_integrada = (DataQualityCheck(fato_alfabetizacao_municipio, "fato_alfabetizacao_municipio",
                                     "silver", output_dir=QUALITY)
                    .check_chave_composta(["ano", "id_municipio", "rede"])
                    .check_not_null(["nome_municipio", "sigla_uf", "nome_regiao"])
                    .check_range("taxa_alfabetizacao", 0, 100)
                    .check_range("meta_taxa_2030", 0, 100)
                    .check_referential_integrity("sigla_uf", dim_uf, "sigla_uf"))
    relatorios_silver.append(dq_integrada.relatorio(imprimir=False))
    dq_integrada.aprovar_ou_falhar()

    # Se rede=3 não fosse Municipal, quase tudo aqui divergiria. A tolerância de
    # 0,1 p.p. existe porque as duas tabelas arredondam diferente em 2023.
    cruzamento = integrada_municipal.merge(metas_referencia, on=["ano", "id_municipio"],
                                           how="inner").dropna()
    desvio = (cruzamento["taxa_alfabetizacao"] - cruzamento["taxa_fonte_meta"]).abs()
    pct_divergente = (desvio > 0.1).mean() * 100

    dq_cruzado = (DataQualityCheck(integrada_municipal, "cross_check_rede_municipal",
                                   "silver", output_dir=QUALITY)
                  .check_cross_table(metas_referencia, ["ano", "id_municipio"],
                                     "taxa_alfabetizacao", "taxa_fonte_meta",
                                     tolerancia=0.1, severidade="WARNING")
                  .check_expectativa(
                      "mapeamento_rede:municipal",
                      pct_divergente < 1.0,
                      f"{pct_divergente:.3f}% de divergência entre as duas fontes oficiais "
                      f"(limite para considerar o mapeamento válido: 1%)",
                      dimensao="CONSISTENCIA"))
    relatorios_silver.append(dq_cruzado.relatorio())
    dq_cruzado.aprovar_ou_falhar()

    # O que sobra é inconsistência da própria fonte, e fica registrado.
    divergentes = cruzamento[desvio > 0.1].copy()
    if len(divergentes):
        divergentes["diferenca_pp"] = (divergentes["taxa_alfabetizacao"]
                                       - divergentes["taxa_fonte_meta"]).round(2)
        monitor.registrar_alerta(
            "WARNING",
            f"{len(divergentes)} município(s) com divergência entre a tabela de indicador e a "
            f"tabela de metas — reportar ao produtor do dado (Inep/Base dos Dados).",
        )
        print("\nDIVERGÊNCIAS ENTRE FONTES OFICIAIS (indicador × metas):")
        display(divergentes[["ano", "id_municipio", "taxa_alfabetizacao",
                             "taxa_fonte_meta", "diferenca_pp"]])

    # ─── (b) Reconstrução do indicador a partir dos alunos ───
    # Com a regra correta (média ponderada, apenas os avaliados), o resultado
    # tem que ser o mesmo número publicado pelo Inep.
    avaliados = fato_aluno[fato_aluno["avaliado"]]

    def ponderar(grupo):
        return pd.Series({
            "taxa_reconstruida": np.average(grupo["alfabetizado"],
                                            weights=grupo["peso_aluno"]) * 100,
            "media_reconstruida": np.average(grupo["proficiencia_portugues"],
                                             weights=grupo["peso_aluno"]),
            "alunos": len(grupo),
        })

    reconstruido = (avaliados.groupby(["ano", "id_municipio", "rede"], observed=True)
                             .apply(ponderar, include_groups=False)
                             .reset_index())
    reconstruido = com_chaves_inteiras(reconstruido, ["ano", "id_municipio", "rede"])

    comparacao = reconstruido.merge(
        com_chaves_inteiras(fato_indicador_municipio[["ano", "id_municipio", "rede",
                                                      "taxa_alfabetizacao", "media_portugues"]],
                            ["ano", "id_municipio", "rede"]),
        on=["ano", "id_municipio", "rede"], how="inner")

    erro_taxa  = (comparacao["taxa_reconstruida"] - comparacao["taxa_alfabetizacao"]).abs()
    erro_media = (comparacao["media_reconstruida"] - comparacao["media_portugues"]).abs()
    dentro_da_tolerancia = (erro_taxa < 0.05).mean() * 100
    erro_nacional = abs(
        np.average(comparacao["taxa_reconstruida"], weights=comparacao["alunos"])
        - np.average(comparacao["taxa_alfabetizacao"], weights=comparacao["alunos"]))

    dq_aluno = (DataQualityCheck(fato_aluno, "fato_aluno", "silver", output_dir=QUALITY)
                .check_row_count(1_000)
                .check_chave_composta(["ano", "id_aluno"])
                .check_not_null(["id_aluno", "ano", "id_municipio"])
                .check_range("proficiencia_portugues", 200, 1000)
                .check_range("peso_aluno", 0, 1000)
                .check_referential_integrity("id_municipio", dim_municipio, "id_municipio")
                .check_expectativa(
                    "reconstrucao:taxa_nacional",
                    erro_nacional <= 0.5,
                    f"diferença de {erro_nacional:.4f} p.p. entre o indicador reconstruído dos "
                    f"microdados e o publicado (tolerância: 0,5 p.p.)",
                    dimensao="CONSISTENCIA")
                .check_expectativa(
                    "reconstrucao:erro_por_municipio",
                    erro_taxa.mean() <= 0.5,
                    f"erro médio de {erro_taxa.mean():.4f} p.p. em {len(comparacao):,} combinações; "
                    f"{dentro_da_tolerancia:.1f}% ficam dentro de 0,05 p.p.".replace(",", "."),
                    dimensao="CONSISTENCIA")
                .check_expectativa(
                    "reconstrucao:media_proficiencia",
                    erro_media.mean() <= 0.5,
                    f"erro médio de {erro_media.mean():.4f} pontos na média de proficiência",
                    dimensao="CONSISTENCIA"))
    relatorios_silver.append(dq_aluno.relatorio())
    dq_aluno.aprovar_ou_falhar()

    metricas["linhas_entrada"] = sum(len(silver[t]) for t in silver)

print("\nPAINEL DE QUALIDADE — CAMADA SILVER")
display(consolidar_relatorios(relatorios_silver))

In [ ]:
# ============================================================
# ETAPA 4.7 — INVENTÁRIO DA CAMADA SILVER
# ============================================================
df_inventario_silver = pd.DataFrame(INVENTARIO_SILVER)
df_inventario_silver["tamanho"] = df_inventario_silver["bytes"].apply(formatar_bytes)

(SILVER / f"_inventario_{RUN_ID}.json").write_text(
    json.dumps(INVENTARIO_SILVER, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"SILVER — {len(df_inventario_silver)} tabelas | "
      f"{df_inventario_silver['linhas'].sum():,} linhas | "
      f"{formatar_bytes(df_inventario_silver['bytes'].sum())}".replace(",", "."))
display(df_inventario_silver[["tabela", "linhas", "colunas", "particionada_por", "tamanho"]])

---
## Etapa 5 — Monitoramento da execução

Todas as etapas foram instrumentadas. O painel abaixo responde ao que um plantão precisa saber:
rodou, demorou quanto, perdeu dado, gerou alerta.

| Métrica | Equivalente na AWS |
|---|---|
| duração por etapa | CloudWatch Metrics |
| linhas de entrada / saída / rejeitadas | métrica customizada (`PutMetricData`) |
| taxa de rejeição acima do limite | CloudWatch Alarm → SNS |
| falha de etapa | Step Functions `Catch` → SNS |
| relatório JSON da execução | S3 + Athena (histórico consultável) |

Medir **taxa de rejeição** importa tanto quanto medir sucesso: um job que roda em 40 segundos e
descarta 30% das linhas passa em qualquer monitoramento ingênuo.

In [ ]:
# ============================================================
# ETAPA 5.1 — PAINEL DE EXECUÇÃO E RELATÓRIO FINAL
# ============================================================
monitor.imprimir_painel()
caminho_relatorio = monitor.salvar()

# Consolidação da qualidade das duas camadas
painel_qualidade = consolidar_relatorios(relatorios_bronze + relatorios_silver)
score_medio = painel_qualidade["score_%"].mean()

print(f"\nSCORE MÉDIO DE QUALIDADE: {score_medio:.1f}%")
print(f"Relatório de execução : {caminho_relatorio}")
print(f"Relatórios de qualidade: {QUALITY}/")
display(painel_qualidade)

In [ ]:
# ============================================================
# ETAPA 5.2 — PRÉVIA DO QUE A CAMADA GOLD VAI RESPONDER
# ============================================================
# Ainda não é a Gold (ela é construída em PySpark no Notebook 03), mas já
# conseguimos demonstrar que a Silver integrada responde às perguntas de negócio.

recorte = fato_alfabetizacao_municipio[
    (fato_alfabetizacao_municipio["ano"] == fato_alfabetizacao_municipio["ano"].max())
    & (fato_alfabetizacao_municipio["rede"] == 3)
]

print(f"Ano de referência: {recorte['ano'].max()}  |  Municípios: {recorte['id_municipio'].nunique():,}"
      .replace(",", "."))
print(f"Taxa média de alfabetização (rede municipal): {recorte['taxa_alfabetizacao'].mean():.2f}%")
print(f"Municípios que atingiram a meta do ano: "
      f"{recorte['atingiu_meta_ano'].mean() * 100:.1f}%")
print(f"Municípios já acima da meta nacional de 2030 (80%): "
      f"{recorte['alcancou_meta_2030'].mean() * 100:.1f}%\n")

por_regiao = (recorte.groupby("nome_regiao")
              .agg(municipios=("id_municipio", "nunique"),
                   taxa_media=("taxa_alfabetizacao", "mean"),
                   gap_medio_2030=("gap_meta_2030", "mean"),
                   pct_na_meta=("atingiu_meta_ano", "mean"))
              .round(2)
              .sort_values("taxa_media", ascending=False))
por_regiao["pct_na_meta"] = (por_regiao["pct_na_meta"] * 100).round(1)
display(por_regiao)

---
## Encerramento

Artefatos gerados:

```
data/lake/bronze/     8 entidades (Parquet, particionado por data de ingestão)
data/lake/silver/     11 tabelas  (Parquet, particionado por ano)
data/lake/quality/    relatórios de qualidade por tabela
data/lake/monitoring/ relatório da execução

src/quality/data_quality.py   framework de qualidade (usado também pelos jobs Glue)
src/utils/monitoring.py       monitoramento
src/utils/lakeio.py           leitura/escrita padronizada do lake
```

**Próximo:** o Notebook 02 faz a ingestão em streaming com Kafka e integra os eventos nesta mesma
camada Silver.